Prima fase, importazione librerire

In [4]:
import pandas as pd
import numpy as np
import re
import torch

print(f"PyTorch Version: {torch.__version__}")
# Impostiamo la GPU come device principale (se disponibile)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sto usando: {device}")

PyTorch Version: 2.7.1+cu118
Sto usando: cuda


Preprocesso il dataset

In [5]:
def preprocess_sentence(w):
    # Tutto in minuscolo e togliamo gli spazi all'inizio e alla fine
    w = str(w).lower().strip()

    # Stacchiamo la punteggiatura dalle parole per considerarla come token separati
    # Es: "ciao." diventa "ciao ."
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)

    # MAGIA: Teniamo le lettere base, i numeri, la punteggiatura e TUTTI gli accenti tipici dei dialetti/italiano
    # Includiamo à, è, é, ì, í, ò, ó, ù, ú, ö, ü
    w = re.sub(r"[^a-zàèéìíòóùúöü0-9?.!,¿']+", " ", w)
    
    w = w.strip()
    
    # Aggiungiamo i token speciali per far capire alla rete quando inizia e finisce la frase
    w = '<sos> ' + w + ' <eos>'
    return w

# Facciamo un test veloce per vedere se funziona
test_phrase = "L'è svèelt com ü fölmen." # Dal tuo file berg.txt
print("Prima:", test_phrase)
print("Dopo: ", preprocess_sentence(test_phrase))

Prima: L'è svèelt com ü fölmen.
Dopo:  <sos> l'è svèelt com ü fölmen . <eos>


Caricamento del file


In [6]:
# Leggiamo il file ignorando la terza colonna (le fonti)
df = pd.read_csv("berg.txt", sep="\t", header=None, usecols=[0, 1], names=["Bergamasco", "Italiano"])

# Puliamo eventuali righe vuote
df = df.dropna()

# Applichiamo il preprocessing a entrambe le colonne
df['Bergamasco_pulito'] = df['Bergamasco'].apply(preprocess_sentence)
df['Italiano_pulito'] = df['Italiano'].apply(preprocess_sentence)

# Mostriamo le prime 5 righe
display(df.head())

,Bergamasco,Italiano,Bergamasco_pulito,Italiano_pulito
0,Visiù,Visione,<sos> visiù <eos>,<sos> visione <eos>
1,In mès a sbarlafüs e mes-ciotade,In mezzo ad oggetti e cianfrusaglie,<sos> in mès a sbarlafüs e mes ciotade <eos>,<sos> in mezzo ad oggetti e cianfrusaglie <eos>
2,ó troàt la sò curuna del rosare...,ho trovato la sua corona del rosario...,<sos> ó troàt la sò curuna del rosare . . . <eos>,<sos> ho trovato la sua corona del rosario . ....
3,la giràa söi nodèi a belasi,la girava piano piano sopra le nocchne,<sos> la giràa söi nodèi a belasi <eos>,<sos> la girava piano piano sopra le nocchne <...
4,insèma l'mururà,insieme al mormorare,<sos> insèma l'mururà <eos>,<sos> insieme al mormorare <eos>


In [ ]:
class Vocabulary:
    def __init__(self):
        # Inizializziamo i dizionari con i 4 token speciali di base
        self.word2index = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.index2word = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.n_words = 4  # Contatore delle parole (partiamo da 4 perché abbiamo già i token speciali)

    def add_sentence(self, sentence):
        # Dividiamo la frase in singole parole (sugli spazi)
        for word in sentence.split(' '):
            self.add_word(word)

    def add_word(self, word):
        # Se la parola non è mai stata vista, la aggiungiamo e le diamo un numero (ID)
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1

# Creiamo le due "scatole" vuote per le due lingue
vocab_berg = Vocabulary()
vocab_ita = Vocabulary()

# Leggiamo riga per riga dal tuo file excel (DataFrame) e popoliamo i vocabolari
for frase in df['Bergamasco_pulito']:
    vocab_berg.add_sentence(frase)

for frase in df['Italiano_pulito']:
    vocab_ita.add_sentence(frase)

print(f"Grandezza vocabolario Bergamasco: {vocab_berg.n_words} parole uniche")
print(f"Grandezza vocabolario Italiano: {vocab_ita.n_words} parole uniche")

# Facciamo un test divertente: vediamo che numero ha assegnato alla parola "visiù" (se esiste nel tuo file)
parola_test = "visiù"
if parola_test in vocab_berg.word2index:
    print(f"L'ID della parola '{parola_test}' è: {vocab_berg.word2index[parola_test]}")